# NetGuard IDS - ML Model Training & API

Please upload KDDTrain+.csv to the Colab workspace before running.

In [ ]:
!pip install pyngrok flask flask-cors scikit-learn pandas joblib

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score
import joblib
import warnings

warnings.filterwarnings('ignore')

print('Loading dataset...')
columns = ['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent',
           'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root',
           'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
           'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate',
           'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
           'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
           'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label']
try:
    df = pd.read_csv('KDDTrain+.csv', header=None)
    if len(df.columns) == 43:
        df.columns = columns + ['difficulty_level']
        df = df.drop('difficulty_level', axis=1)
    elif len(df.columns) == 42:
        df.columns = columns
    else:
        df = pd.read_csv('KDDTrain+.csv')
except FileNotFoundError:
    print('ERROR: KDDTrain+.csv not found! Please upload it.')

In [ ]:
if 'df' in locals():
    print('Preprocessing dataset...')
    categorical_cols = ['protocol_type', 'service', 'flag']
    label_encoders = {}
    for col in categorical_cols:
        le = LabelEncoder()
        df[col] = pd.Series(le.fit_transform(df[col].astype(str)))
        label_encoders[col] = le
    
    X = df.drop('label', axis=1)
    y = df['label']
    y_binary = y.apply(lambda x: 'Normal' if x == 'normal' else 'Attack')
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    print('Training Random Forest Classifier...')
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_scaled, y_binary)
    
    preds = rf_model.predict(X_scaled)
    print(f'Training Accuracy: {accuracy_score(y_binary, preds) * 100:.2f}%')
    
    joblib.dump(rf_model, 'rf_ids_model.pkl')
    joblib.dump(scaler, 'scaler.pkl')
    joblib.dump(label_encoders, 'label_encoders.pkl')
    joblib.dump(columns[:-1], 'features.pkl')
    
    print('Models saved. Downloading...')
    try:
        from google.colab import files
        files.download('rf_ids_model.pkl')
        files.download('scaler.pkl')
        files.download('label_encoders.pkl')
    except ImportError:
        pass

In [ ]:
import os
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import threading
import pandas as pd
import joblib

app = Flask(__name__)
CORS(app)

try:
    rf_model = joblib.load('rf_ids_model.pkl')
    scaler = joblib.load('scaler.pkl')
    label_encoders = joblib.load('label_encoders.pkl')
    EXPECTED_FEATURES = joblib.load('features.pkl')
except:
    print('Models not found. Train the model first.')
    EXPECTED_FEATURES = []

def preprocess_input(df_input):
    categorical_cols = ['protocol_type', 'service', 'flag']
    for col in categorical_cols:
        if col in df_input.columns:
            le = label_encoders[col]
            classes = list(le.classes_)
            df_input[col] = df_input[col].apply(lambda x: x if x in classes else classes[0])
            df_input[col] = le.transform(df_input[col].astype(str))
    for col in EXPECTED_FEATURES:
        if col not in df_input.columns:
            df_input[col] = 0
    df_input = df_input[EXPECTED_FEATURES]
    return scaler.transform(df_input)

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.json
        features = data.get('features', {})
        if isinstance(features, list):
            df_input = pd.DataFrame([features], columns=EXPECTED_FEATURES[:len(features)])
        else:
            df_input = pd.DataFrame([features])
        
        X_scaled = preprocess_input(df_input)
        pred = rf_model.predict(X_scaled)[0]
        probs = rf_model.predict_proba(X_scaled)[0]
        return jsonify({'prediction': pred, 'confidence': round(max(probs) * 100, 2), 'attack_type': 'DoS' if pred == 'Attack' else 'None' })
    except Exception as e:
        return jsonify({'error': str(e)}), 400

@app.route('/upload', methods=['POST'])
def upload():
    try:
        if 'file' not in request.files:
            return jsonify({'error': 'No file part'}), 400
        file = request.files['file']
        df_input = pd.read_csv(file)
        X_scaled = preprocess_input(df_input.copy())
        preds = rf_model.predict(X_scaled)
        probs = rf_model.predict_proba(X_scaled)
        results = [{'row': i, 'prediction': p, 'confidence': round(max(probs[i])*100, 2)} for i, p in enumerate(preds)]
        return jsonify({'results': results})
    except Exception as e:
        return jsonify({'error': str(e)}), 400

port = 5000
public_url = ngrok.connect(port).public_url
print('\n' + '='*60 + f'\n* NGROK API URL: {public_url}\n' + '='*60 + '\n')
app.run(port=port)
